# 04 - Red neuronal con PyTorch

Se implementa una red neuronal equivalente a la de TensorFlow para comparar tecnologías con el mismo problema.


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

BASE_DIR = Path.cwd().parent
df = pd.read_csv(BASE_DIR / "datos" / "dataset_fraude_yape.csv")
df["fecha_hora"] = pd.to_datetime(df["fecha_hora"])
df["hora"] = df["fecha_hora"].dt.hour
df["dia_semana"] = df["fecha_hora"].dt.dayofweek

X = df.drop(columns=["id_transaccion", "fecha_hora", "puntaje_riesgo", "nivel_riesgo", "fraude"])
y = df["fraude"].astype(int)
X = pd.get_dummies(X, columns=["producto"], dtype=float).fillna(0).astype(float)

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1,1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)

loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=256, shuffle=True)
print("Variables:", X_train.shape[1])


In [ ]:
class RedFraude(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(n_features, 128), nn.ReLU(),
            nn.BatchNorm1d(128), nn.Dropout(0.30),
            nn.Linear(128, 64), nn.ReLU(),
            nn.BatchNorm1d(64), nn.Dropout(0.25),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1))

    def forward(self, x):
        return self.red(x)

modelo = RedFraude(X_train.shape[1])
normal = int((y_train == 0).sum())
fraude = int((y_train == 1).sum())
pos_weight = torch.tensor([normal / fraude], dtype=torch.float32)

criterio = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizador = torch.optim.Adam(modelo.parameters(), lr=0.001)
print(modelo)


In [ ]:
historial = []
for epoch in range(40):
    modelo.train()
    perdida_total = 0.0
    for xb, yb in loader:
        optimizador.zero_grad()
        salida = modelo(xb)
        perdida = criterio(salida, yb)
        perdida.backward()
        optimizador.step()
        perdida_total += perdida.item()
    perdida_promedio = perdida_total / len(loader)
    historial.append(perdida_promedio)
    if (epoch + 1) % 5 == 0:
        print(f"Época {epoch+1:02d}/40 - Loss: {perdida_promedio:.4f}")


In [ ]:
modelo.eval()
with torch.no_grad():
    probabilidades = torch.sigmoid(modelo(X_test_t)).numpy().ravel()
pred = (probabilidades >= 0.5).astype(int)

metricas = {
    "Accuracy": accuracy_score(y_test, pred),
    "Precision": precision_score(y_test, pred, zero_division=0),
    "Recall": recall_score(y_test, pred, zero_division=0),
    "F1": f1_score(y_test, pred, zero_division=0)
}
print(metricas)
print("\nMatriz de confusión:\n", confusion_matrix(y_test, pred))
print("\n", classification_report(y_test, pred, target_names=["Normal", "Fraude"]))


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(historial)
plt.title("Pérdida de la red neuronal PyTorch")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.show()


In [ ]:
MODELOS_DIR = BASE_DIR / "modelos"
MODELOS_DIR.mkdir(exist_ok=True)
torch.save(modelo.state_dict(), MODELOS_DIR / "modelo_pytorch.pth")
print("Modelo PyTorch guardado.")


## Resultado de referencia

PyTorch obtuvo aproximadamente **96.12% Accuracy, 63.01% Precision, 96.27% Recall y 76.17% F1**. Aunque tuvo mayor Recall, Gradient Boosting obtuvo el mejor F1 y Precision de la comparación final.
